# TxGNN KG Schema — Overview & Usage Guide

This notebook walks through all elements of `manage_db/kg_schema.py`:

1. [Node types & primary ontologies](#1-node-types)
2. [Cross-reference columns](#2-cross-reference-columns)
3. [Relation taxonomy](#3-relation-taxonomy)
4. [Credibility scores](#4-credibility-scores)
5. [Xref resolution helpers](#5-xref-resolution)
6. [Legacy TxGNN mapping](#6-legacy-mapping)
7. [End-to-end pipeline sketch](#7-pipeline)


In [ ]:
import sys
sys.path.insert(0, '..')   # repo root

from manage_db.kg_schema import (
    NodeType, NODE_TYPES, NODE_XREF_COLUMNS,
    Relation, RelationKind, RELATIONS, RELATION_BY_NAME,
    RELATIONS_BY_SOURCE, RELATIONS_BY_TARGET,
    Credibility,
    XrefResolution, XREF_RESOLUTION, XREF_BY_COLUMN,
    LEGACY_NODE_TYPE_MAP, LEGACY_RELATION_MAP, LEGACY_RELATION_FLIP,
    EDGE_PARQUET_COLUMNS, NODE_PARQUET_PRIMARY_COLUMN,
    relation_names, node_type_names, relations_between,
    xref_columns_for, primary_ontology_for,
)
import pandas as pd

---
## 1 — Node types

Each node type has exactly **one** primary ontology.  All node Parquet files
store the primary ID in a column called `id`.

In [ ]:
rows = []
for nt, info in NODE_TYPES.items():
    rows.append({
        "node_type": nt.value,
        "primary_ontology": info.primary_ontology,
        "id_format": info.id_format,
        "bionty_registry": info.bionty_registry or "—",
        "example_id": info.example_id,
    })

pd.DataFrame(rows).set_index("node_type")

In [ ]:
# Convenience helpers
print("All node type names:", node_type_names())
print("Primary ontology for disease:", primary_ontology_for(NodeType.DISEASE))
print("Primary ontology for gene:   ", primary_ontology_for(NodeType.GENE))

---
## 2 — Cross-reference columns

Every node Parquet file contains `id` (primary) plus the `xref_columns` for
that node type.  These columns are nullable strings.

**Example — Disease node columns:**
```
id            EFO:0000305          ← primary (always present)
mondo_id      MONDO:0007254        ← cross-reference (nullable)
omim_id       114480               ← cross-reference (nullable)
doid_id       DOID:1612            ← cross-reference (nullable)
icd10_code    C50.9                ← cross-reference (nullable)
mesh_id       D001943              ← cross-reference (nullable)
hp_id         (null)
```

In [ ]:
# Show all xref columns per node type
for nt in NodeType:
    cols = xref_columns_for(nt)
    if cols:
        print(f"{nt.value:12s}  →  {', '.join(cols)}")
    else:
        print(f"{nt.value:12s}  →  (no xrefs)")

In [ ]:
# Example: build the full column list for a gene Parquet file
gene_parquet_columns = [NODE_PARQUET_PRIMARY_COLUMN] + list(xref_columns_for(NodeType.GENE))
print("Gene Parquet required columns:", gene_parquet_columns)

### Simulated node record

Below is what a row in `data/kg/nodes/disease.parquet` looks like:

In [ ]:
import pandas as pd

disease_columns = [NODE_PARQUET_PRIMARY_COLUMN] + list(xref_columns_for(NodeType.DISEASE)) + ["name"]
example_disease = pd.DataFrame([{
    "id":         "EFO:0000305",
    "mondo_id":   "MONDO:0007254",
    "omim_id":    "114480",
    "doid_id":    "DOID:1612",
    "icd10_code": "C50.9",
    "mesh_id":    "D001943",
    "hp_id":      None,
    "name":       "breast carcinoma",
}], columns=disease_columns)

example_disease

---
## 3 — Relation taxonomy

76 canonical directed relation types grouped by `RelationKind`.

In [ ]:
# Summary by kind
from collections import Counter

kind_counts = Counter(r.kind.value for r in RELATIONS)
pd.Series(kind_counts, name="count").sort_values(ascending=False)

In [ ]:
# All relation names
print("Total relations:", len(RELATIONS))
for name in relation_names():
    print(" ", name)

In [ ]:
# Look up a relation by name
rel = RELATION_BY_NAME["molecule_treats_disease"]
print(f"name:   {rel.name}")
print(f"source: {rel.source.value}")
print(f"target: {rel.target.value}")
print(f"kind:   {rel.kind.value}")
print(f"direct: {rel.direct}")
print(f"notes:  {rel.notes}")

In [ ]:
# Relations between two specific node types
rels = relations_between(NodeType.DISEASE, NodeType.GENE)
print("Disease → Gene relations:")
for r in rels:
    print(f"  {r.name:40s}  direct={r.direct}  kind={r.kind.value}")

In [ ]:
# All relations that target Disease
print("Relations targeting Disease:")
for r in RELATIONS_BY_TARGET.get(NodeType.DISEASE, []):
    print(f"  {r.source.value:12s} → disease   [{r.name}]")

In [ ]:
# Direct vs indirect relations
direct = [r for r in RELATIONS if r.direct]
indirect = [r for r in RELATIONS if not r.direct]
print(f"Direct:   {len(direct)}")
print(f"Indirect: {len(indirect)}")

---
## 4 — Credibility scores

Every edge in a Parquet file carries a `credibility` column (int 1–3).

In [ ]:
for c in Credibility:
    print(f"{c.value}  {c.name:20s}  — {c.__doc__ or ''}")

# Usage: assign credibility when ingesting edges
print("\nExample:")
print("  ChEMBL indication  →", int(Credibility.ESTABLISHED_FACT))
print("  Single GWAS hit    →", int(Credibility.SINGLE_EVIDENCE))

### Edge Parquet schema

Every edge file has at minimum these columns:

In [ ]:
pd.DataFrame(EDGE_PARQUET_COLUMNS, columns=["column", "description"])

In [ ]:
# Example edge row
example_edge = pd.DataFrame([{
    "x_id":             "CHEMBL941",
    "x_type":           "molecule",
    "y_id":             "EFO:0000305",
    "y_type":           "disease",
    "relation":         "molecule_treats_disease",
    "display_relation": "treats",
    "source":           "ChEMBL",
    "credibility":      3,
}])
example_edge

---
## 5 — Xref resolution helpers

`XREF_RESOLUTION` describes how to go from an external ID *to* the canonical
primary ID.  Each entry maps one `xref_column` to a resolution recipe
(description + optional API URL template).

In [ ]:
# All xref resolutions for Disease
disease_xrefs = [x for x in XREF_RESOLUTION if x.node_type == NodeType.DISEASE]
for x in disease_xrefs:
    print(f"  {x.xref_column:15s}  ({x.from_namespace})")
    print(f"    → {x.description}")
    if x.url_template:
        print(f"    URL: {x.url_template}")

In [ ]:
# Look up a specific xref resolution
res = XREF_BY_COLUMN[("drugbank_id", NodeType.MOLECULE)]
print("DrugBank → ChEMBL resolution:")
print(f"  from_namespace: {res.from_namespace}")
print(f"  description:    {res.description}")
print(f"  URL template:   {res.url_template}")

# Build an actual URL for DB01267 (Imatinib / Gleevec)
drugbank_id = "DB01267"
url = res.url_template.format(id=drugbank_id)
print(f"  → Resolved URL: {url}")

In [ ]:
# Summary table of all xref resolutions
xref_rows = [{
    "node_type": x.node_type.value,
    "xref_column": x.xref_column,
    "from_namespace": x.from_namespace,
    "has_api": bool(x.url_template),
} for x in XREF_RESOLUTION]
pd.DataFrame(xref_rows)

---
## 6 — Legacy TxGNN mapping

The original TxGNN KG uses different node type strings and relation names.
`LEGACY_NODE_TYPE_MAP` and `LEGACY_RELATION_MAP` translate them to the new schema.

In [ ]:
# Node type mapping
pd.DataFrame(
    [(k, v.value) for k, v in LEGACY_NODE_TYPE_MAP.items()],
    columns=["legacy_type", "new_type"]
)

In [ ]:
# Relation mapping
pd.DataFrame(
    [(k, v) for k, v in LEGACY_RELATION_MAP.items()],
    columns=["legacy_relation", "canonical_relation"]
)

In [ ]:
# Relations whose source/target must be flipped during migration
print("Relations that require x/y swap on migration:")
for r in sorted(LEGACY_RELATION_FLIP):
    canonical = LEGACY_RELATION_MAP.get(r, "(not in map)")
    print(f"  {r:35s}  →  {canonical}")

---
## 7 — End-to-end pipeline sketch

Below is a minimal template showing how to go from a raw edge CSV (using legacy
TxGNN naming) to a compliant edge Parquet file with canonical IDs.

In [ ]:
import pandas as pd
from manage_db.kg_schema import (
    LEGACY_NODE_TYPE_MAP, LEGACY_RELATION_MAP, LEGACY_RELATION_FLIP,
    RELATION_BY_NAME, Credibility, NODE_PARQUET_PRIMARY_COLUMN,
)

# ── Simulated raw TxGNN edge data ────────────────────────────────────────
raw_edges = pd.DataFrame([
    # x_id       x_type         y_id                y_type     relation
    ("imatinib", "drug",        "chronic myeloid leukemia", "disease",  "indication"),
    ("BRCA2",    "gene/protein","breast cancer",    "disease",  "biomarker"),
    ("BRCA2",    "gene/protein","DNA repair",       "biological_process", "bioprocess_protein"),
], columns=["x_id", "x_type", "y_id", "y_type", "relation"])

print("Raw edges:")
display(raw_edges)

# ── Step 1: translate node types ─────────────────────────────────────────
raw_edges["x_type"] = raw_edges["x_type"].map(LEGACY_NODE_TYPE_MAP).apply(lambda x: x.value if x else None)
raw_edges["y_type"] = raw_edges["y_type"].map(LEGACY_NODE_TYPE_MAP).apply(lambda x: x.value if x else None)

# ── Step 2: flip edges that have reversed direction in legacy data ────────
flip_mask = raw_edges["relation"].isin(LEGACY_RELATION_FLIP)
raw_edges.loc[flip_mask, ["x_id", "x_type", "y_id", "y_type"]] = (
    raw_edges.loc[flip_mask, ["y_id", "y_type", "x_id", "x_type"]].values
)

# ── Step 3: translate relation names ─────────────────────────────────────
raw_edges["relation"] = raw_edges["relation"].map(LEGACY_RELATION_MAP)

# ── Step 4: add required Parquet columns ─────────────────────────────────
raw_edges["display_relation"] = raw_edges["relation"].apply(
    lambda r: RELATION_BY_NAME[r].notes if r in RELATION_BY_NAME else r
)
raw_edges["source"] = "TxGNN-legacy"
raw_edges["credibility"] = int(Credibility.ESTABLISHED_FACT)

print("\nNormalised edges (ready to write as Parquet):")
display(raw_edges)

In [ ]:
# ── Validating a node DataFrame against schema ────────────────────────────
from manage_db.kg_schema import xref_columns_for, NodeType

def validate_node_df(df: pd.DataFrame, node_type: NodeType) -> list[str]:
    """Return a list of schema violations for a node DataFrame."""
    errors = []
    required = [NODE_PARQUET_PRIMARY_COLUMN] + list(xref_columns_for(node_type))
    for col in required:
        if col not in df.columns:
            errors.append(f"Missing required column: '{col}'")
    if NODE_PARQUET_PRIMARY_COLUMN in df.columns and df[NODE_PARQUET_PRIMARY_COLUMN].isna().any():
        errors.append("Primary ID column 'id' contains null values")
    return errors


# Example: well-formed gene DataFrame
good_genes = pd.DataFrame([{
    "id":           "ENSG00000139618",
    "ncbi_gene_id": "675",
    "hgnc_id":      "HGNC:1101",
    "uniprot_id":   "P51587",
    "gene_name":    "BRCA2",
}])

errors = validate_node_df(good_genes, NodeType.GENE)
print("Validation errors (good df):", errors or "none ✓")

# Example: missing primary ID
bad_genes = good_genes.drop(columns=["id"])
errors = validate_node_df(bad_genes, NodeType.GENE)
print("Validation errors (bad df): ", errors)

---
## Quick-reference cheat sheet

| Task | Code |
|------|------|
| Primary ontology for a node type | `primary_ontology_for(NodeType.DISEASE)` |
| Xref columns for a node type | `xref_columns_for(NodeType.MOLECULE)` |
| All relations from source | `RELATIONS_BY_SOURCE[NodeType.MOLECULE]` |
| Relations between two types | `relations_between(NodeType.DISEASE, NodeType.GENE)` |
| Lookup relation by name | `RELATION_BY_NAME["molecule_treats_disease"]` |
| Xref resolution recipe | `XREF_BY_COLUMN[("drugbank_id", NodeType.MOLECULE)]` |
| Translate legacy node type | `LEGACY_NODE_TYPE_MAP["gene/protein"]` |
| Translate legacy relation | `LEGACY_RELATION_MAP["indication"]` |
| Credibility for curated DB | `int(Credibility.ESTABLISHED_FACT)` → 3 |